# RECOMENDADOR HÍBRIDO DE PELÍCULAS TMDB (2010–2025): CONTENIDO Y CALIDAD PONDERADA
## Fase 3: Núcleo algorítmico, eficiencia y programación orientada a objetos

---

- **Asignatura:** MCDI500 Programación para la Ciencia de Datos
- **Profesor:** Omar Salinas Silva
- **Evaluación:** Sumativa 2, avance de la Fase 3
- **Integrantes del Grupo 4:**
  1. Daniela Baeza Rocha
  2. Juan González Elgueta
  3. Elías Manríquez Armijo
  4. Yestxander Rada Gutiérrez

---

## Propósito de esta entrega

**Pregunta del proyecto (la misma de F1, F2 y el informe):** ¿en qué medida un recomendador basado en la similitud de contenido (géneros, productoras e idioma original) y ponderado por calidad (*Weighted Rating*) y popularidad entrega, para una película de TMDB 2010–2025, recomendaciones coherentes con ella en género e idioma y respaldadas por un volumen suficiente de votos?

Esta entrega corrige el pendiente que dejó la Sumativa 1: el pipeline de clases de F3 reescribía la sopa de metadatos sin quitar la puntuación de las productoras, así que la equivalencia que probaba no era con el F2 real. Ahora cada paso de la Fase 2 es una clase hija de `Transformador` que llama a las funciones de `src/preparacion.py`, y el cuaderno comprueba que el pipeline reproduce **exactamente** `data/processed/movies_procesado.csv` y la matriz `movies_bow_matrix.npz`. Incluye la partición sin fuga de datos, la validación con casos límite y excepciones, y el guardado de resultados. El recomendador, la recursividad y las mediciones de eficiencia se agregan en entregas siguientes.

**Módulos:** `src/carga.py`, `src/transformadores.py`, `src/pipeline.py` y `src/preparacion.py`. **Pruebas:** `tests/test_transformadores.py` y `tests/test_preparacion.py`.

**Origen del código.** Los transformadores que reproducen F2, la comprobación contra `movies_procesado.csv` y la carga con otro separador parten de la propuesta de Daniela (PR #43).

## 0. Configuración
Única celda a editar: rutas, semilla, columnas y parámetros. Todo lo demás se construye desde ella.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Localiza la raíz del proyecto aunque Jupyter se abra en otra carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pipeline.py").exists())
sys.path.insert(0, str(RAIZ))

SEMILLA = 42
FRACCION_PRUEBA = 0.20
RUTA_DATOS = RAIZ / "data" / "raw" / "movies_raw.csv"
DIR_PROCESADO = RAIZ / "data" / "processed"
DIR_RESULTADOS = RAIZ / "F3" / "results"

COLUMNAS_ESPERADAS = [
    "movie_id", "title", "release_date", "status", "budget", "revenue", "runtime",
    "genres", "production_companies", "production_countries", "spoken_languages",
    "original_language", "vote_average", "vote_count", "popularity", "adult",
]
COLUMNAS_NUMERICAS = ["budget", "revenue", "runtime", "vote_average", "vote_count", "popularity"]

N_RECOMENDACIONES = 10        # recomendaciones por película, como en el criterio de F1
N_CONSULTAS_AJUSTE = 2_000    # películas de entrenamiento con que se eligen los pesos
# Línea base aleatoria de F1 (sección 3): el recomendador debe superar las tres cifras
LINEA_BASE_F1 = {"Coherencia de género": 0.418, "Coherencia de idioma": 0.366, "Respaldo de votos": 0.250}

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["font.size"] = 10
pd.set_option("display.max_colwidth", 60)

### Verificación del entorno
Se importa cada librería, se compara su versión con la fijada en `requirements.txt` y se registran la rama y el commit, para asociar cada resultado a una versión del repositorio (igual que en F1 y F2).

In [2]:
import subprocess
from importlib import metadata

LIBRERIAS = [("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "sklearn"),
             ("scipy", "scipy"), ("matplotlib", "matplotlib")]


def leer_versiones_fijadas(ruta):
    """Devuelve {paquete: versión} a partir de las líneas `paquete==versión` de requirements.txt."""
    fijadas = {}
    for linea in Path(ruta).read_text(encoding="utf-8").splitlines():
        if "==" in linea and not linea.lstrip().startswith("#"):
            nombre, version = linea.strip().split("==")
            fijadas[nombre.lower().replace("_", "-")] = version
    return fijadas


fijadas = leer_versiones_fijadas(RAIZ / "requirements.txt")
print(f"Python {sys.version.split()[0]}\n")
print(f"{'paquete':<14}{'instalada':<12}{'requirements':<14}estado")
for paquete, modulo in LIBRERIAS:
    __import__(modulo)  # falla aquí si la librería no está instalada
    instalada = metadata.version(paquete)
    declarada = fijadas.get(paquete, "no fijada")
    print(f"{paquete:<14}{instalada:<12}{declarada:<14}{'OK' if instalada == declarada else 'DIFIERE'}")


def git(*args):
    try:
        return subprocess.run(["git", *args], capture_output=True, text=True, check=True, cwd=RAIZ).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return "no disponible"


print(f"\nRama: {git('rev-parse', '--abbrev-ref', 'HEAD')} | commit: {git('rev-parse', '--short', 'HEAD')}")

Python 3.13.0

paquete       instalada   requirements  estado
numpy         2.5.4       2.5.4         OK
pandas        3.1.1       3.1.1         OK


scikit-learn  1.9.1       1.9.1         OK
scipy         1.18.1      1.18.1        OK
matplotlib    3.11.2      3.11.2        OK

Rama: feat/f3-transformadores-f2 | commit: b584638


## 1. Carga con verificación de esquema
`src/carga.py` falla con un mensaje claro si falta el archivo o una columna declarada, y fuerza a número las columnas numéricas: un texto suelto las convertiría en columnas de objetos sin aviso. Además se exige que `movie_id` sea único, porque el recomendador lo usa como llave.

In [3]:
from src.carga import cargar

df = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS, COLUMNAS_NUMERICAS)
assert df["movie_id"].is_unique, "movie_id debe identificar una sola película"

print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas | movie_id único: {df['movie_id'].is_unique}")
pd.DataFrame({
    "tipo": df[COLUMNAS_NUMERICAS].dtypes.astype(str),
    "nulos": df[COLUMNAS_NUMERICAS].isna().sum(),
    "ceros": (df[COLUMNAS_NUMERICAS] == 0).sum(),
})

17,978 filas x 16 columnas | movie_id único: True


,tipo,nulos,ceros
budget,int64,0,12463
revenue,int64,0,11156
runtime,int64,0,46
vote_average,float64,0,0
vote_count,int64,0,0
popularity,float64,0,0


**Interpretación:** las 17.978 películas y las 16 columnas declaradas están presentes, las columnas numéricas quedan como números y `movie_id` es único. Ninguna columna numérica tiene nulos, pero hay ceros que significan «sin dato»: 12.463 en `budget`, 11.156 en `revenue` y 46 en `runtime`. El pipeline los trata como en F2: en `runtime` se imputan por la mediana del género principal y en `budget` y `revenue` se dejan como ausentes, porque el recomendador no usa esas columnas.

## 2. Pipeline con clases: reproducir la Fase 2
Cada paso de F2 es una clase hija de `Transformador` (`src/transformadores.py`):

- **Herencia.** `Transformador` define el contrato común: `ajustar()` valida las columnas de entrada, delega el aprendizaje en `aprender()` y marca el paso como ajustado; `transformar()` exige el ajuste, vuelve a validar el esquema y trabaja sobre una copia. Las hijas solo implementan lo que cambia.
- **Polimorfismo.** `Pipeline` (`src/pipeline.py`) recorre los pasos llamando a los mismos métodos, sin saber de qué clase es cada uno: `DuracionPorGenero` aprende medianas por género, `CalidadPonderada` aprende C y m, `EscaladorMinMax` aprende mínimo y rango.
- **Encapsulamiento.** Lo aprendido se guarda en `_parametros` y se expone con la propiedad `parametros`, que devuelve una copia; `nombre` es una propiedad sin *setter*. Un paso sin ajustar no puede transformar.
- **Reutilización.** Las reglas de limpieza no se reescriben: `ConstructorSopa` llama a `imputar_metadatos`, `clean_genres`, `clean_text_feature` y `construir_sopa` de `src/preparacion.py`, y `CalidadPonderada` a `calculate_weighted_rating`. Esto resuelve el pendiente que dejó la Sumativa 1: la versión anterior de F3 reescribía la sopa sin quitar la puntuación de las productoras y se comparaba contra esa misma reescritura, no contra el F2 real.

In [4]:
from src.pipeline import Pipeline
from src.preparacion import diccionario_procesado
from src.transformadores import (CalidadPonderada, CerosComoAusentes, ConstructorSopa, DuracionPorGenero,
                                 EscaladorMinMax, LogNaturalPlus1, Transformador)

COLUMNAS_SALIDA = diccionario_procesado()["columna"].tolist()   # mismas columnas y orden que F2


def construir_pipeline():
    return Pipeline([
        DuracionPorGenero(),                                       # runtime_clean
        CerosComoAusentes("budget", "budget_clean"),
        CerosComoAusentes("revenue", "revenue_clean"),
        ConstructorSopa(),                                         # genres/companies/lang_clean y metadata_soup
        CalidadPonderada(),                                        # weighted_rating (aprende C y m)
        LogNaturalPlus1("popularity", "popularity_log"),
        EscaladorMinMax("weighted_rating", "score_quality_norm"),
        EscaladorMinMax("popularity_log", "score_popularity_norm"),
    ])


pipe = construir_pipeline()
resumen = pipe.resumen()
# Herencia: todos los pasos son Transformador; polimorfismo: cada uno redefine aprender() y aplicar()
resumen["hereda de Transformador"] = [isinstance(p, Transformador) for p in pipe.pasos_ejecutados()]
resumen["redefine aprender/aplicar"] = [
    type(p).aprender is not Transformador.aprender and type(p).aplicar is not Transformador.aplicar
    for p in pipe.pasos_ejecutados()
]
print(pipe)
resumen

Pipeline(8 pasos, sin ajustar)


,orden,paso,clase,hereda de Transformador,redefine aprender/aplicar
0,1,DuracionPorGenero(runtime),DuracionPorGenero,True,True
1,2,CerosComoAusentes(budget),CerosComoAusentes,True,True
2,3,CerosComoAusentes(revenue),CerosComoAusentes,True,True
3,4,ConstructorSopa(metadata_soup),ConstructorSopa,True,True
4,5,CalidadPonderada(vote_average),CalidadPonderada,True,True
5,6,LogNaturalPlus1(popularity),LogNaturalPlus1,True,True
6,7,EscaladorMinMax(weighted_rating),EscaladorMinMax,True,True
7,8,EscaladorMinMax(popularity_log),EscaladorMinMax,True,True


In [5]:
# Encapsulamiento: el estado interno solo se lee, y lo que se entrega es una copia
paso = CalidadPonderada().ajustar(df)
copia = paso.parametros
copia["m"] = 0                                   # modificar la copia no altera el paso
assert paso.parametros["m"] > 0

try:
    paso.nombre = "otro"                         # nombre es una propiedad sin setter
except AttributeError as e:
    print(f"OK AttributeError: no se puede asignar 'nombre' ({e})")

print(f"Parámetros aprendidos por {paso.nombre}: C = {paso.parametros['C']:.4f}, m = {paso.parametros['m']:.2f}")

OK AttributeError: no se puede asignar 'nombre' (property 'nombre' of 'CalidadPonderada' object has no setter)
Parámetros aprendidos por CalidadPonderada(vote_average): C = 6.3295, m = 427.75


**Interpretación:** los ocho pasos heredan de `Transformador` y cada uno redefine `aprender()` y `aplicar()`; el `Pipeline` los ejecuta en orden sin tratamientos especiales. La copia defensiva funciona: modificar el diccionario que entrega `parametros` no altera el paso, y asignar `nombre` lanza `AttributeError`. Sobre el catálogo completo, `CalidadPonderada` aprende C = 6,3295 (nota media) y m = 427,75 votos (percentil 75), los mismos valores de F2.

### 2.1 Equivalencia con la Fase 2
El pipeline con clases debe dar **exactamente** el conjunto que exporta F2 (`data/processed/movies_procesado.csv`) y la misma matriz *Bag of Words* (`movies_bow_matrix.npz`). Para eso se ajusta sobre el catálogo completo, como hizo F2. Si no coincidiera, la refactorización habría cambiado el comportamiento.

In [6]:
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

catalogo_f2 = construir_pipeline().ajustar(df).transformar(df)[COLUMNAS_SALIDA]
referencia = pd.read_csv(DIR_PROCESADO / "movies_procesado.csv")
referencia["companies_clean"] = referencia["companies_clean"].fillna("")   # el CSV lee el texto vacío como NaN

pd.testing.assert_frame_equal(catalogo_f2.reset_index(drop=True), referencia,
                              check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
matriz_f3 = CountVectorizer().fit_transform(catalogo_f2["metadata_soup"])
matriz_f2 = sparse.load_npz(DIR_PROCESADO / "movies_bow_matrix.npz")
assert matriz_f3.shape == matriz_f2.shape and (matriz_f3 != matriz_f2).nnz == 0

print(f"OK: tabla {catalogo_f2.shape[0]:,} x {catalogo_f2.shape[1]} idéntica a la de F2 (tolerancia 1e-12)")
print(f"OK: matriz {matriz_f3.shape[0]:,} x {matriz_f3.shape[1]:,} con {matriz_f3.nnz:,} valores no nulos, idéntica a la de F2")

OK: tabla 17,978 x 16 idéntica a la de F2 (tolerancia 1e-12)
OK: matriz 17,978 x 13,766 con 102,902 valores no nulos, idéntica a la de F2


**Interpretación:** el pipeline con clases reproduce la Fase 2 sin diferencias: la tabla de 17.978 × 16 coincide con `movies_procesado.csv` dentro de una tolerancia de 1e-12 (la que admite el redondeo del CSV) y la matriz *Bag of Words* de 17.978 × 13.766 tiene sus 102.902 valores no nulos en las mismas posiciones que `movies_bow_matrix.npz`. La refactorización no cambió el comportamiento, así que todo lo que sigue se apoya en el mismo conjunto que se documentó y evaluó en la Sumativa 1.

## 3. Partición entrenamiento / prueba y ajuste sin fuga de datos
Se separa **antes** de ajustar: las medianas de duración, C y m del *Weighted Rating*, los mínimos y rangos de escalado y el vocabulario se aprenden solo con entrenamiento y se aplican a prueba.

In [7]:
prueba = df.sample(frac=FRACCION_PRUEBA, random_state=SEMILLA)
entrenamiento = df.drop(prueba.index)
assert set(entrenamiento["movie_id"]).isdisjoint(prueba["movie_id"])
assert len(entrenamiento) + len(prueba) == len(df)

pipe = construir_pipeline().ajustar(entrenamiento)
parametros_antes = {p.nombre: p.parametros for p in pipe.pasos_ejecutados()}
entrenamiento_final = pipe.transformar(entrenamiento)[COLUMNAS_SALIDA]
prueba_final = pipe.transformar(prueba)[COLUMNAS_SALIDA]
# Transformar prueba no debe cambiar lo aprendido
assert parametros_antes == {p.nombre: p.parametros for p in pipe.pasos_ejecutados()}

print(f"entrenamiento: {len(entrenamiento):,} | prueba: {len(prueba):,}")
calidad = pipe.pasos_ejecutados()[4].parametros
print(f"Weighted Rating aprendido en entrenamiento: C = {calidad['C']:.4f}, m = {calidad['m']:.2f} votos")
pd.DataFrame({
    conjunto: {f"{c} [min, max]": f"[{d[c].min():.3f}, {d[c].max():.3f}]"
               for c in ["score_quality_norm", "score_popularity_norm", "runtime_clean"]}
    for conjunto, d in [("entrenamiento", entrenamiento_final), ("prueba", prueba_final)]
})

entrenamiento: 14,382 | prueba: 3,596
Weighted Rating aprendido en entrenamiento: C = 6.3270, m = 421.00 votos


,entrenamiento,prueba
"score_quality_norm [min, max]","[0.000, 1.000]","[-0.023, 1.012]"
"score_popularity_norm [min, max]","[0.000, 1.000]","[0.007, 0.970]"
"runtime_clean [min, max]","[2.000, 585.000]","[2.000, 260.000]"


**Interpretación:** con la semilla 42 quedan 14.382 películas de entrenamiento y 3.596 de prueba, sin `movie_id` en común. Los parámetros aprendidos en entrenamiento (C = 6,3270, m = 421 votos) son cercanos a los del catálogo completo (6,3295 y 427,75), señal de que la muestra es representativa. Transformar prueba no altera lo aprendido. Como el mínimo y el rango vienen de entrenamiento, en prueba `score_quality_norm` llega a [−0,023; 1,012]: hay películas con un *Weighted Rating* algo fuera del rango visto. No se recorta, para no ocultarlo; el efecto en el puntaje es de unas centésimas.

## 4. Validación: caso normal, casos límite y excepciones
Las pruebas unitarias de `tests/` (`python -m pytest`) cubren cada clase y función por separado; aquí se valida el flujo completo sobre los datos del proyecto.

In [8]:
from tempfile import TemporaryDirectory


# --- CASO NORMAL ---
assert entrenamiento_final["metadata_soup"].str.len().gt(0).all()
assert entrenamiento_final[["score_quality_norm", "score_popularity_norm"]].apply(lambda s: s.between(0, 1)).all().all()
assert entrenamiento_final["runtime_clean"].gt(0).all()
assert entrenamiento_final["movie_id"].tolist() == entrenamiento["movie_id"].tolist()   # conserva orden e identidad
print("Normal: sopa no vacía, puntuaciones en [0, 1], duraciones positivas y mismo orden de películas.")

# --- CASOS LÍMITE ---
una = pd.DataFrame({"genres": [None], "production_companies": [None], "original_language": [None],
                    "runtime": [90.0], "budget": [0.0], "revenue": [0.0],
                    "vote_average": [7.0], "vote_count": [100], "popularity": [0.0]})
salida = construir_pipeline().ajustar(una).transformar(una)
assert salida["metadata_soup"].iloc[0] == "unknown unknown"   # nulos imputados como en F2
assert salida["score_quality_norm"].iloc[0] == 0.0            # rango 0: el divisor cae a 1, sin división por cero
print("Límite: una sola película con nulos y columnas constantes no rompe el pipeline.")

with TemporaryDirectory() as carpeta:   # el mismo catálogo con otro separador
    ruta = Path(carpeta) / "catalogo_punto_y_coma.csv"
    prueba.head(10).to_csv(ruta, sep=";", index=False)
    diez = cargar(ruta, COLUMNAS_ESPERADAS, COLUMNAS_NUMERICAS)
pd.testing.assert_frame_equal(pipe.transformar(diez)[COLUMNAS_SALIDA], prueba_final.head(10).reset_index(drop=True),
                              check_dtype=False)
print("Límite: un CSV separado por ';' se carga y produce el mismo resultado.")

# --- EXCEPCIONES ---
def debe_fallar(tipo, accion):
    try:
        accion()
    except tipo as e:
        print(f"OK {tipo.__name__}: {e}")
    else:
        raise AssertionError(f"Debía lanzar {tipo.__name__}")


debe_fallar(FileNotFoundError, lambda: cargar(RAIZ / "data" / "raw" / "no_existe.csv", COLUMNAS_ESPERADAS))
debe_fallar(RuntimeError, lambda: construir_pipeline().transformar(entrenamiento))
debe_fallar(KeyError, lambda: pipe.transformar(prueba.drop(columns=["genres"])))
debe_fallar(TypeError, lambda: Pipeline().agregar("no soy un transformador"))
debe_fallar(ValueError, lambda: LogNaturalPlus1("popularity").ajustar_transformar(pd.DataFrame({"popularity": [-1.0]})))
debe_fallar(ValueError, lambda: CalidadPonderada().ajustar(pd.DataFrame({"vote_average": [5.0], "vote_count": [0]})))

Normal: sopa no vacía, puntuaciones en [0, 1], duraciones positivas y mismo orden de películas.
Límite: una sola película con nulos y columnas constantes no rompe el pipeline.
Límite: un CSV separado por ';' se carga y produce el mismo resultado.
OK FileNotFoundError: No se encontró el archivo: C:\Users\Yestx\repos\Personal$2\Magister CDIA\entregas\mcdi500\proyecto-tmdb-mcdi500\data\raw\no_existe.csv
OK RuntimeError: Pipeline: hay que ajustar antes de transformar.
OK KeyError: "DuracionPorGenero(runtime): faltan columnas de entrada ['genres']."
OK TypeError: Se esperaba un Transformador y se recibió str.
OK ValueError: LogNaturalPlus1(popularity): log1p no admite valores negativos.
OK ValueError: CalidadPonderada(vote_average): se requiere C finito y un percentil de votos positivo.


**Interpretación:** el flujo completo pasa las comprobaciones del caso normal y de los casos límite. Una película con todos sus metadatos nulos produce la sopa `unknown unknown`, igual que en F2, y una columna constante no provoca una división por cero. Un CSV separado por punto y coma, frecuente en datos públicos chilenos, se carga y produce el mismo resultado. Cada uso incorrecto falla con la excepción prevista y un mensaje que dice qué corregir: archivo inexistente, pipeline sin ajustar, columna faltante en prueba, paso que no es un `Transformador`, `log1p` con negativos y catálogo sin votos para calcular m.

## 8. Guardar el resultado y su configuración
El conjunto procesado no sirve sin el registro de cómo se obtuvo: se guardan los conjuntos de entrenamiento y prueba con las mismas columnas que F2, los parámetros aprendidos, los pesos elegidos y las mediciones. `data/processed/movies_procesado.csv` (F2) no se toca.

In [9]:
import json

DIR_PROCESADO.mkdir(parents=True, exist_ok=True)
entrenamiento_final.to_csv(DIR_PROCESADO / "entrenamiento_f3.csv", index=False)
prueba_final.to_csv(DIR_PROCESADO / "prueba_f3.csv", index=False)

parametros = {
    "semilla": SEMILLA,
    "fraccion_prueba": FRACCION_PRUEBA,
    "pasos": {p.nombre: p.parametros for p in pipe.pasos_ejecutados()},
}
(DIR_PROCESADO / "parametros_f3.json").write_text(
    json.dumps(parametros, indent=2, ensure_ascii=False, default=float), encoding="utf-8")

# Comprobación: lo guardado se vuelve a leer sin sorpresas
leido = pd.read_csv(DIR_PROCESADO / "entrenamiento_f3.csv")
assert list(leido.columns) == COLUMNAS_SALIDA and len(leido) == len(entrenamiento_final)
assert leido["movie_id"].tolist() == entrenamiento_final["movie_id"].tolist()
np.testing.assert_allclose(leido["score_quality_norm"], entrenamiento_final["score_quality_norm"])
print("Guardado:", sorted(p.relative_to(RAIZ).as_posix() for p in DIR_PROCESADO.glob("*_f3.*")))

Guardado: ['data/processed/entrenamiento_f3.csv', 'data/processed/parametros_f3.json', 'data/processed/prueba_f3.csv']


## Conclusiones

1. **El pipeline de clases reproduce exactamente la Fase 2** (tabla de 17.978 × 16 y matriz de 17.978 × 13.766 idénticas) llamando a las funciones de `src/preparacion.py`. Se cumple lo que pidió la retroalimentación de la Sumativa 1: las funciones de F2 viven en módulos propios y F3 parte de ellas.
2. **Sin fuga de datos.** C, m, las medianas de duración y los rangos de escalado se aprenden solo con las 14.382 películas de entrenamiento; transformar las 3.596 de prueba no los altera.
3. **Validación.** Caso normal, casos límite y 6 excepciones sobre los datos del proyecto, más las pruebas de `tests/test_transformadores.py`.

**Para las entregas siguientes.** Quedan en `data/processed/` los conjuntos de entrenamiento y prueba con las mismas columnas que F2 y `parametros_f3.json` con los parámetros aprendidos. Sobre ellos se construye el recomendador, la recursividad y las mediciones de eficiencia.